> **Chapter Map:** This notebook is the companion for **Chapter 4** (Transport Limitations).

**Prerequisites:** You should be comfortable with: (1) PFR design equations from NB3, (2) first-order reaction kinetics from NB2, (3) the concept of diffusion coefficients.

# NB4: Transport Limitations and Effectiveness Factors

## Learning Objectives

After completing this notebook, you will be able to:

1. Calculate the Thiele modulus for slab and spherical catalyst geometries
2. Implement and plot effectiveness factor functions for both geometries
3. Identify kinetic-control and diffusion-control regimes from log-log plots
4. Visualize intraparticle concentration profiles and connect them to effectiveness factor values
5. Apply the Weisz--Prater criterion to diagnose diffusion limitations from observable quantities
6. Distinguish external film mass transfer from internal pore diffusion using the Biot number and overall effectiveness factor
7. Compare diffusion mechanisms in zeolites (configurational, activated) and nanotubes (Knudsen, enhanced)
8. Integrate diffusion-limited kinetics into a PFR model and assess the impact of pellet size on conversion

## Background

In heterogeneous catalysis, reactants must diffuse through the porous structure of catalyst pellets to reach active sites. When the intrinsic reaction rate is fast relative to diffusion, reactant concentration drops significantly inside the pellet, and the **observed rate** is lower than the **intrinsic rate**. The **effectiveness factor** quantifies this penalty.

### Key Equations

**Thiele Modulus** -- compares reaction rate to diffusion rate:

$$\phi_{\text{slab}} = L \sqrt{\frac{k}{D_{\text{eff}}}}, \qquad \phi_{\text{sphere}} = R_p \sqrt{\frac{k}{D_{\text{eff}}}}$$

**Effectiveness Factor** -- ratio of actual rate to rate at surface conditions:

$$\eta_{\text{slab}} = \frac{\tanh\phi}{\phi}, \qquad \eta_{\text{sphere}} = \frac{3}{\phi^2}\left(\phi\coth\phi - 1\right)$$

**Observed Rate**:

$$r_{\text{obs}} = \eta \cdot r_{\text{intrinsic}}(C_{A,s})$$

**Weisz--Prater Criterion** (uses observable quantities only):

$$N_{WP} = \frac{r_{\text{obs}} \cdot R_p^2}{C_{A,s} \cdot D_{\text{eff}}} = \eta \cdot \phi^2$$

- $N_{WP} < 0.3$: negligible diffusion limitation
- $N_{WP} > 3$: severe diffusion limitation

**Concentration Profile Inside a Slab** (first-order, symmetric about center):

$$\frac{C_A(x)}{C_{A,s}} = \frac{\cosh(\phi \cdot x/L)}{\cosh(\phi)}$$

where $x$ is measured from the center ($x=0$) to the surface ($x=L$).

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Publication-quality plot defaults
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constant
R = 8.314  # Gas constant, J/(mol*K)

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

print("Setup complete. Libraries imported successfully.")

> **Safety Note.** Laboratory diagnosis of diffusion limitations (the "crushing test") generates fine catalyst dust that may contain toxic metals (Ni, Co, Cr, V). Always perform crushing in a fume hood and wear a dust mask. High-temperature reactions on porous catalysts can also generate hot spots leading to thermal runaway — monitor bed temperature at multiple points.

---
## Part 1: Thiele Modulus

The Thiele modulus is a dimensionless number that compares the characteristic rate of reaction to the characteristic rate of diffusion inside a catalyst pellet.

- **Slab** (half-thickness $L$): $\phi = L\sqrt{k/D_{\text{eff}}}$
- **Sphere** (radius $R_p$): $\phi = R_p\sqrt{k/D_{\text{eff}}}$

When $\phi \ll 1$, diffusion is fast compared to reaction (kinetic control). When $\phi \gg 1$, reaction is fast compared to diffusion (diffusion control).

In [ ]:
def thiele_modulus_slab(L, k, D_eff):
    """Calculate Thiele modulus for a flat slab.

    Parameters
    ----------
    L : float or array-like
        Half-thickness of the slab (m)
    k : float
        First-order rate constant (s^-1)
    D_eff : float
        Effective diffusivity (m^2/s)

    Returns
    -------
    float or ndarray
        Thiele modulus (dimensionless)

    Notes
    -----
    phi = L * sqrt(k / D_eff)
    """
    L = np.asarray(L)
    return L * np.sqrt(k / D_eff)


def thiele_modulus_sphere(R_p, k, D_eff):
    """Calculate Thiele modulus for a sphere.

    Parameters
    ----------
    R_p : float or array-like
        Pellet radius (m)
    k : float
        First-order rate constant (s^-1)
    D_eff : float
        Effective diffusivity (m^2/s)

    Returns
    -------
    float or ndarray
        Thiele modulus (dimensionless)

    Notes
    -----
    phi = R_p * sqrt(k / D_eff)
    """
    R_p = np.asarray(R_p)
    return R_p * np.sqrt(k / D_eff)


# --- Verify with a known case ---
# From Example 4.1 in the lecture notes: k = 1.0 s^-1, D_eff = 1e-6 m^2/s
k_test = 1.0        # s^-1
D_eff_test = 1e-6   # m^2/s

R_p_values = [0.1e-3, 1.0e-3, 5.0e-3]  # m
print("Thiele Modulus Verification (sphere)")
print("=" * 45)
print(f"{'R_p (mm)':>10} | {'phi':>8}")
print("-" * 45)
for R_p in R_p_values:
    phi = thiele_modulus_sphere(R_p, k_test, D_eff_test)
    print(f"{R_p*1000:>10.1f} | {phi:>8.1f}")

### How Each Parameter Affects the Thiele Modulus

Since $\phi = R_p\sqrt{k/D_{\text{eff}}}$, the Thiele modulus increases with:
- Larger pellet size $R_p$
- Faster intrinsic kinetics $k$
- Lower effective diffusivity $D_{\text{eff}}$

Let us visualize the individual effect of each parameter.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: Effect of pellet radius ---
R_p_range = np.linspace(0.05e-3, 10e-3, 200)  # 0.05 to 10 mm
k_ref = 1.0       # s^-1
D_eff_ref = 1e-6   # m^2/s

phi_Rp = thiele_modulus_sphere(R_p_range, k_ref, D_eff_ref)

ax1 = axes[0]
ax1.plot(R_p_range * 1000, phi_Rp, color=COLORS['blue'], linewidth=2.5)
ax1.axhline(y=0.3, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.axhline(y=3.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.text(0.3, 0.35, r'$\phi = 0.3$ (kinetic limit)', fontsize=9, color='gray')
ax1.text(0.3, 3.3, r'$\phi = 3$ (diffusion limit)', fontsize=9, color='gray')
ax1.set_xlabel(r'Pellet Radius, $R_p$ (mm)')
ax1.set_ylabel(r'Thiele Modulus, $\phi$')
ax1.set_title(f'Effect of $R_p$\n($k$ = {k_ref} s$^{{-1}}$, '
              f'$D_{{eff}}$ = {D_eff_ref:.0e} m$^2$/s)')

# --- Panel 2: Effect of rate constant ---
k_range = np.logspace(-2, 2, 200)  # 0.01 to 100 s^-1
R_p_ref = 1e-3     # 1 mm

phi_k = thiele_modulus_sphere(R_p_ref, k_range, D_eff_ref)

ax2 = axes[1]
ax2.loglog(k_range, phi_k, color=COLORS['orange'], linewidth=2.5)
ax2.axhline(y=0.3, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax2.axhline(y=3.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax2.set_xlabel(r'Rate Constant, $k$ (s$^{-1}$)')
ax2.set_ylabel(r'Thiele Modulus, $\phi$')
ax2.set_title(f'Effect of $k$\n($R_p$ = {R_p_ref*1000:.0f} mm, '
              f'$D_{{eff}}$ = {D_eff_ref:.0e} m$^2$/s)')

# --- Panel 3: Effect of effective diffusivity ---
D_eff_range = np.logspace(-8, -4, 200)  # 1e-8 to 1e-4 m^2/s

phi_D = thiele_modulus_sphere(R_p_ref, k_ref, D_eff_range)

ax3 = axes[2]
ax3.loglog(D_eff_range, phi_D, color=COLORS['green'], linewidth=2.5)
ax3.axhline(y=0.3, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax3.axhline(y=3.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax3.set_xlabel(r'Effective Diffusivity, $D_{eff}$ (m$^2$/s)')
ax3.set_ylabel(r'Thiele Modulus, $\phi$')
ax3.set_title(f'Effect of $D_{{eff}}$\n($R_p$ = {R_p_ref*1000:.0f} mm, '
              f'$k$ = {k_ref} s$^{{-1}}$)')
ax3.invert_xaxis()  # Lower D_eff -> higher phi, so invert for intuition

plt.tight_layout()
plt.show()

print("Key insight: phi increases linearly with R_p but only as sqrt(k).")
print("Reducing pellet size is the most direct way to lower phi.")

---
## Part 2: Effectiveness Factor -- Slab Geometry

The effectiveness factor for a flat slab with first-order kinetics is:

$$\eta_{\text{slab}} = \frac{\tanh\phi}{\phi}$$

**Limiting behavior:**
- $\phi \to 0$: $\eta \to 1$ (kinetic control)
- $\phi \to \infty$: $\eta \to 1/\phi$ (diffusion control, since $\tanh\phi \to 1$)

In [ ]:
def effectiveness_slab(phi):
    """Calculate effectiveness factor for slab (flat plate) geometry.

    Parameters
    ----------
    phi : float or array-like
        Thiele modulus (dimensionless)

    Returns
    -------
    float or ndarray
        Effectiveness factor eta, range (0, 1]

    Notes
    -----
    eta = tanh(phi) / phi

    Limiting cases:
    - phi << 1: eta ~ 1 (kinetic control)
    - phi >> 1: eta ~ 1/phi (diffusion control)
    """
    phi = np.asarray(phi, dtype=float)
    # Handle phi = 0 to avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        eta = np.where(phi < 1e-10, 1.0, np.tanh(phi) / phi)
    return eta


# Test at a few values
test_phis = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0]
print("Slab Effectiveness Factor")
print("=" * 40)
print(f"{'phi':>8} | {'eta':>10} | {'1/phi':>10}")
print("-" * 40)
for p in test_phis:
    eta = effectiveness_slab(p)
    print(f"{p:>8.2f} | {eta:>10.4f} | {1/p:>10.4f}")

In [ ]:
# Log-log plot of eta vs phi for the slab
phi_range = np.logspace(-2, 2, 500)
eta_slab = effectiveness_slab(phi_range)

fig, ax = plt.subplots(figsize=(10, 7))

# Main curve
ax.loglog(phi_range, eta_slab, color=COLORS['blue'], linewidth=2.5,
          label=r'$\eta_{\text{slab}} = \tanh(\phi)/\phi$')

# Asymptotic limits
ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)
phi_asympt = np.logspace(0.3, 2, 100)
ax.loglog(phi_asympt, 1.0 / phi_asympt, color=COLORS['vermillion'],
          linestyle='--', linewidth=2,
          label=r'Asymptote: $\eta = 1/\phi$')

# Shade kinetic and diffusion regimes
ax.axvspan(0.01, 0.3, alpha=0.10, color=COLORS['green'],
           label=r'Kinetic regime ($\phi < 0.3$, $\eta \approx 1$)')
ax.axvspan(3.0, 100, alpha=0.10, color=COLORS['orange'],
           label=r'Diffusion regime ($\phi > 3$, $\eta \approx 1/\phi$)')

# Mark boundary values
for phi_mark, label_text in [(0.3, r'$\phi=0.3$'), (3.0, r'$\phi=3$')]:
    eta_mark = effectiveness_slab(phi_mark)
    ax.plot(phi_mark, eta_mark, 'ko', markersize=7)
    ax.annotate(f'{label_text}\n$\\eta$ = {eta_mark:.3f}',
                xy=(phi_mark, eta_mark),
                xytext=(phi_mark * 2.5, eta_mark * 1.5),
                fontsize=10,
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

ax.set_xlabel(r'Thiele Modulus, $\phi$')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title('Effectiveness Factor vs Thiele Modulus (Slab Geometry)')
ax.set_xlim(0.01, 100)
ax.set_ylim(0.008, 1.5)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

**Concept Check:** A colleague claims: "If η = 0.5, it means only the outer half of the pellet is being used." Is this statement correct? Why or why not?

*Think about what the concentration profile looks like at η = 0.5 before reading the answer below.*

<details><summary>Answer</summary>

**Not correct.** The effectiveness factor η = 0.5 means the *overall* rate is half of what it would be at uniform surface concentration — but the entire pellet volume still participates in reaction to some degree. The concentration profile is smooth (exponential-like), not a step function. Even at the center, there is some reactant present and some reaction occurring. The "outer half" interpretation confuses the integrated average rate with a spatial cutoff.

</details>

---
## Part 3: Effectiveness Factor -- Sphere Geometry

For a spherical pellet with first-order kinetics, the effectiveness factor is:

$$\eta_{\text{sphere}} = \frac{3}{\phi^2}\left(\phi\coth\phi - 1\right)$$

**Limiting behavior:**
- $\phi \to 0$: $\eta \to 1$
- $\phi \to \infty$: $\eta \to 3/\phi$ (diffusion control; note the factor of 3 vs 1 for the slab)

In [ ]:
def effectiveness_sphere(phi):
    """Calculate effectiveness factor for spherical geometry.

    Parameters
    ----------
    phi : float or array-like
        Thiele modulus (dimensionless)

    Returns
    -------
    float or ndarray
        Effectiveness factor eta, range (0, 1]

    Notes
    -----
    eta = 3 * (phi * coth(phi) - 1) / phi^2

    Limiting cases:
    - phi << 1: eta ~ 1 (kinetic control)
    - phi >> 1: eta ~ 3/phi (diffusion control)
    """
    phi = np.asarray(phi, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        coth_phi = 1.0 / np.tanh(phi)
        eta = 3 * (phi * coth_phi - 1) / phi**2
        eta = np.where(phi < 1e-10, 1.0, eta)
    return eta


# Verify with lecture note Example 4.1
print("Sphere Effectiveness Factor Verification")
print("(cf. Example 4.1 in the lecture notes)")
print("=" * 50)
print(f"{'R_p (mm)':>10} | {'phi':>6} | {'eta (exact)':>12} | {'eta (notes)':>12}")
print("-" * 50)
for R_p_mm, phi_expected, eta_expected in [(0.1, 0.1, 0.999),
                                           (1.0, 1.0, 0.94),
                                           (5.0, 5.0, 0.48)]:
    phi_val = phi_expected
    eta_val = effectiveness_sphere(phi_val)
    print(f"{R_p_mm:>10.1f} | {phi_val:>6.1f} | {eta_val:>12.4f} | {eta_expected:>12.3f}")

In [ ]:
# Compare slab and sphere on the same log-log plot
phi_range = np.logspace(-2, 2, 500)
eta_slab_vals = effectiveness_slab(phi_range)
eta_sphere_vals = effectiveness_sphere(phi_range)

fig, ax = plt.subplots(figsize=(10, 7))

# Main curves
ax.loglog(phi_range, eta_slab_vals, color=COLORS['blue'],
          linewidth=2.5, linestyle='-',
          label=r'Slab: $\eta = \tanh(\phi)/\phi$')
ax.loglog(phi_range, eta_sphere_vals, color=COLORS['orange'],
          linewidth=2.5, linestyle='--',
          label=r'Sphere: $\eta = 3(\phi\coth\phi - 1)/\phi^2$')

# Asymptotic limits
phi_large = np.logspace(0.5, 2, 100)
ax.loglog(phi_large, 1.0 / phi_large, color=COLORS['blue'],
          linestyle=':', linewidth=1.5, alpha=0.6,
          label=r'Slab asymptote: $1/\phi$')
ax.loglog(phi_large, 3.0 / phi_large, color=COLORS['orange'],
          linestyle=':', linewidth=1.5, alpha=0.6,
          label=r'Sphere asymptote: $3/\phi$')

# Shade regimes
ax.axvspan(0.01, 0.3, alpha=0.08, color=COLORS['green'])
ax.axvspan(3.0, 100, alpha=0.08, color=COLORS['vermillion'])
ax.text(0.05, 0.012, 'Kinetic\ncontrol', fontsize=11, ha='center',
        color=COLORS['green'], fontweight='bold')
ax.text(30, 0.012, 'Diffusion\ncontrol', fontsize=11, ha='center',
        color=COLORS['vermillion'], fontweight='bold')
ax.text(1.0, 0.012, 'Transition', fontsize=11, ha='center',
        color='gray', fontweight='bold')

# Horizontal reference at eta = 1
ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel(r'Thiele Modulus, $\phi$')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title('Effectiveness Factor: Slab vs Sphere')
ax.set_xlim(0.01, 100)
ax.set_ylim(0.008, 1.5)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("At the same phi, the sphere has HIGHER eta than the slab.")
print("In the diffusion-limited regime: eta_sphere ~ 3/phi vs eta_slab ~ 1/phi.")
print("The sphere\'s smaller volume-to-surface-area ratio (R_p/3 vs L) means")
print("interior points are, on average, closer to the surface.")
print(f"\nExample at phi = 5:")
print(f"  Slab:   eta = {effectiveness_slab(5.0):.4f}")
print(f"  Sphere: eta = {effectiveness_sphere(5.0):.4f}")

---
## Part 4: Concentration Profiles Inside a Slab

For a slab of half-thickness $L$ with first-order kinetics, the dimensionless concentration profile is:

$$\frac{C_A(x)}{C_{A,s}} = \frac{\cosh(\phi \cdot x/L)}{\cosh(\phi)}$$

where $x$ is the distance from the center ($x = 0$) to the surface ($x = L$). At the surface ($x = L$), the concentration equals $C_{A,s}$; at the center ($x = 0$), the concentration is lowest.

Plotting these profiles for different $\phi$ values shows *why* the effectiveness factor decreases with increasing $\phi$: the reactant simply cannot penetrate to the center of the pellet.

**Concept Check:** For $\phi = 10$, predict: will the concentration at the pellet center be (a) > 50% of surface, (b) 10--50%, (c) 1--10%, or (d) < 1% of surface concentration? Write your prediction, then run the next cell to check.

In [ ]:
def slab_concentration_profile(x_over_L, phi):
    """Dimensionless concentration profile inside a slab.

    Parameters
    ----------
    x_over_L : float or array-like
        Dimensionless position x/L, ranging from 0 (center) to 1 (surface)
    phi : float
        Thiele modulus

    Returns
    -------
    float or ndarray
        C_A(x) / C_A,s (dimensionless)

    Notes
    -----
    C/Cs = cosh(phi * x/L) / cosh(phi)
    """
    x_over_L = np.asarray(x_over_L)
    return np.cosh(phi * x_over_L) / np.cosh(phi)


# Plot profiles for several Thiele modulus values
xi = np.linspace(0, 1, 300)  # Dimensionless position x/L

phi_values = [0.5, 1.0, 2.0, 5.0, 10.0]
colors_profile = [COLORS['blue'], COLORS['green'], COLORS['orange'],
                  COLORS['vermillion'], COLORS['purple']]
linestyles_profile = ['-', '--', '-.', '-', '--']

fig, ax = plt.subplots(figsize=(10, 7))

for phi, color, ls in zip(phi_values, colors_profile, linestyles_profile):
    C_ratio = slab_concentration_profile(xi, phi)
    eta_val = effectiveness_slab(phi)
    ax.plot(xi, C_ratio, color=color, linestyle=ls, linewidth=2.5,
            label=f'$\\phi$ = {phi}, $\\eta$ = {eta_val:.3f}')

# Mark the center value for the largest phi
C_center_10 = slab_concentration_profile(0, 10.0)
ax.annotate(f'$C/C_s$ = {C_center_10:.2e}\n(center, $\\phi$ = 10)',
            xy=(0.02, C_center_10 + 0.02),
            fontsize=10, color=COLORS['purple'])

ax.set_xlabel(r'Dimensionless Position, $x/L$ (0 = center, 1 = surface)')
ax.set_ylabel(r'Dimensionless Concentration, $C_A/C_{A,s}$')
ax.set_title('Concentration Profiles Inside a Catalyst Slab')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  phi = 0.5: Nearly uniform concentration (kinetic control)")
print("  phi = 10:  Concentration drops to near zero at center (diffusion control)")
print("  The steeper the gradient, the lower the effectiveness factor.")

In [ ]:
# Sphere concentration profiles for comparison
# C_A(r)/C_A,s = (R_p/r) * sinh(phi * r/R_p) / sinh(phi)

def sphere_concentration_profile(r_over_R, phi):
    """Dimensionless concentration profile inside a sphere.

    Parameters
    ----------
    r_over_R : float or array-like
        Dimensionless radial position r/R_p (0 = center, 1 = surface)
    phi : float
        Thiele modulus

    Returns
    -------
    float or ndarray
        C_A(r) / C_A,s
    """
    r_over_R = np.asarray(r_over_R, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(
            r_over_R < 1e-10,
            phi / np.sinh(phi),  # L'Hôpital limit at r=0
            np.sinh(phi * r_over_R) / (r_over_R * np.sinh(phi))
        )
    return ratio


# Compare slab vs sphere profiles
rho = np.linspace(0, 1, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for phi, color, ls in zip(phi_values, colors_profile, linestyles_profile):
    # Slab
    C_slab = slab_concentration_profile(rho, phi)
    ax1.plot(rho, C_slab, color=color, linestyle=ls, linewidth=2.5,
             label=f'$\\phi$ = {phi}')
    # Sphere
    C_sphere = sphere_concentration_profile(rho, phi)
    ax2.plot(rho, C_sphere, color=color, linestyle=ls, linewidth=2.5,
             label=f'$\\phi$ = {phi}')

ax1.set_xlabel(r'$x/L$ (0 = center, 1 = surface)')
ax1.set_ylabel(r'$C_A/C_{A,s}$')
ax1.set_title('Slab')
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1.05)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel(r'$r/R_p$ (0 = center, 1 = surface)')
ax2.set_ylabel(r'$C_A/C_{A,s}$')
ax2.set_title('Sphere')
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1.05)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Intraparticle Concentration Profiles: Slab vs Sphere', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("At the same phi, the sphere profile is less depleted at the center")
print("because the spherical geometry has a smaller volume-to-surface ratio.")

---
## Part 5: Sensitivity Analysis -- Restoring Effectiveness

When a catalyst operates in the diffusion-limited regime, we want to restore $\eta$ close to 1. There are three design levers:

1. **Reduce pellet size** $R_p$ (directly reduces $\phi$)
2. **Increase effective diffusivity** $D_{\text{eff}}$ (e.g., larger pores, different support)
3. **Decrease intrinsic rate constant** $k$ (rarely desirable; included for completeness)

Let us start from a diffusion-limited baseline and see how much each lever must change to restore $\eta > 0.95$.

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Baseline: diffusion-limited catalyst.
# Try changing R_p, k, or D_eff to see how phi
# and eta respond.
R_p_base = 3e-3    # 3 mm radius
k_base = 2.0       # s^-1
D_eff_base = 1e-6  # m^2/s
# =============================================

phi_base = thiele_modulus_sphere(R_p_base, k_base, D_eff_base)
eta_base = effectiveness_sphere(phi_base)

print("Baseline Conditions")
print("=" * 50)
print(f"R_p     = {R_p_base*1000:.1f} mm")
print(f"k       = {k_base:.1f} s^-1")
print(f"D_eff   = {D_eff_base:.0e} m^2/s")
print(f"phi     = {phi_base:.2f}")
print(f"eta     = {eta_base:.4f}")
print(f"\nThis catalyst is severely diffusion-limited!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

eta_target = 0.95

# --- Panel 1: Reduce R_p ---
R_p_sweep = np.linspace(0.05e-3, R_p_base, 300)
phi_sweep_Rp = thiele_modulus_sphere(R_p_sweep, k_base, D_eff_base)
eta_sweep_Rp = effectiveness_sphere(phi_sweep_Rp)

ax1 = axes[0]
ax1.plot(R_p_sweep * 1000, eta_sweep_Rp, color=COLORS['blue'], linewidth=2.5)
ax1.axhline(y=eta_target, color='gray', linestyle='--', linewidth=1)
ax1.axhline(y=eta_base, color=COLORS['vermillion'], linestyle=':', linewidth=1.5)

# Find R_p that gives eta = 0.95
idx_95 = np.argmin(np.abs(eta_sweep_Rp - eta_target))
R_p_95 = R_p_sweep[idx_95]
ax1.plot(R_p_95 * 1000, eta_target, 'o', color=COLORS['blue'],
         markersize=10, markeredgecolor='black', markeredgewidth=1.5)
ax1.annotate(f'$R_p$ = {R_p_95*1000:.2f} mm',
             xy=(R_p_95 * 1000, eta_target),
             xytext=(R_p_95 * 1000 + 0.5, eta_target - 0.15),
             fontsize=11, arrowprops=dict(arrowstyle='->', lw=1.2))

ax1.set_xlabel(r'Pellet Radius, $R_p$ (mm)')
ax1.set_ylabel(r'Effectiveness Factor, $\eta$')
ax1.set_title('Strategy 1: Reduce Pellet Size')
ax1.set_ylim(0, 1.05)

# --- Panel 2: Increase D_eff ---
D_eff_sweep = np.logspace(np.log10(D_eff_base), -3, 300)
phi_sweep_D = thiele_modulus_sphere(R_p_base, k_base, D_eff_sweep)
eta_sweep_D = effectiveness_sphere(phi_sweep_D)

ax2 = axes[1]
ax2.semilogx(D_eff_sweep, eta_sweep_D, color=COLORS['green'], linewidth=2.5)
ax2.axhline(y=eta_target, color='gray', linestyle='--', linewidth=1)
ax2.axhline(y=eta_base, color=COLORS['vermillion'], linestyle=':', linewidth=1.5)

idx_95_D = np.argmin(np.abs(eta_sweep_D - eta_target))
D_eff_95 = D_eff_sweep[idx_95_D]
ax2.plot(D_eff_95, eta_target, 'o', color=COLORS['green'],
         markersize=10, markeredgecolor='black', markeredgewidth=1.5)
ax2.annotate(f'$D_{{eff}}$ = {D_eff_95:.1e} m$^2$/s',
             xy=(D_eff_95, eta_target),
             xytext=(D_eff_95 * 3, eta_target - 0.15),
             fontsize=11, arrowprops=dict(arrowstyle='->', lw=1.2))

ax2.set_xlabel(r'Effective Diffusivity, $D_{eff}$ (m$^2$/s)')
ax2.set_ylabel(r'Effectiveness Factor, $\eta$')
ax2.set_title('Strategy 2: Increase Diffusivity')
ax2.set_ylim(0, 1.05)

# --- Panel 3: Decrease k ---
k_sweep = np.linspace(0.001, k_base, 300)
phi_sweep_k = thiele_modulus_sphere(R_p_base, k_sweep, D_eff_base)
eta_sweep_k = effectiveness_sphere(phi_sweep_k)

ax3 = axes[2]
ax3.plot(k_sweep, eta_sweep_k, color=COLORS['orange'], linewidth=2.5)
ax3.axhline(y=eta_target, color='gray', linestyle='--', linewidth=1)
ax3.axhline(y=eta_base, color=COLORS['vermillion'], linestyle=':', linewidth=1.5)

idx_95_k = np.argmin(np.abs(eta_sweep_k - eta_target))
k_95 = k_sweep[idx_95_k]
ax3.plot(k_95, eta_target, 'o', color=COLORS['orange'],
         markersize=10, markeredgecolor='black', markeredgewidth=1.5)
ax3.annotate(f'$k$ = {k_95:.3f} s$^{{-1}}$',
             xy=(k_95, eta_target),
             xytext=(k_95 + 0.3, eta_target - 0.15),
             fontsize=11, arrowprops=dict(arrowstyle='->', lw=1.2))

ax3.set_xlabel(r'Rate Constant, $k$ (s$^{-1}$)')
ax3.set_ylabel(r'Effectiveness Factor, $\eta$')
ax3.set_title('Strategy 3: Decrease Rate Constant')
ax3.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"\nTo achieve eta > {eta_target}:")
print(f"  Reduce R_p from {R_p_base*1000:.1f} mm to {R_p_95*1000:.2f} mm "
      f"(factor of {R_p_base/R_p_95:.1f}x reduction)")
print(f"  Increase D_eff from {D_eff_base:.0e} to {D_eff_95:.1e} m^2/s "
      f"(factor of {D_eff_95/D_eff_base:.0f}x increase)")
print(f"  Decrease k from {k_base:.1f} to {k_95:.3f} s^-1 "
      f"(factor of {k_base/k_95:.0f}x reduction -- not practical!)")
print(f"\nReducing pellet size is the most practical strategy.")

**Concept Check:** An industrial engineer wants to double the catalyst loading in a packed-bed reactor without increasing pressure drop. They propose using pellets with twice the radius (same total mass, fewer pellets). Predict what happens to (a) φ, (b) η, and (c) conversion. Is this a good strategy?

<details><summary>Answer</summary>

**(a)** φ doubles (φ ∝ R_p). **(b)** η decreases significantly — if the original φ = 1.5 (η ≈ 0.76), doubling to φ = 3 gives η ≈ 0.67 for a slab or η ≈ 0.81 for a sphere. **(c)** Conversion drops because the observed rate per unit mass decreases. **Not a good strategy** — the gain from reduced pressure drop is offset by lower catalyst effectiveness. Better approaches: use eggshell catalysts (active material only near the surface) or structured supports (monoliths).

</details>

### Key Takeaways — Part 1 (Internal Diffusion Fundamentals)

1. **Thiele modulus** φ = L√(k/D_eff) is the single dimensionless number controlling diffusion limitation. φ < 0.3 → kinetic control; φ > 3 → diffusion control.
2. **Effectiveness factor** η gives the ratio of actual to ideal rate. For a slab: η = tanh(φ)/φ. For a sphere: η = 3(φ·coth(φ) − 1)/φ².
3. **Sphere vs slab**: At the same φ, the sphere has *higher* η (asymptotically 3/φ vs 1/φ) because its volume-to-surface ratio is smaller.
4. **Design lever**: Reducing pellet size is the most practical way to restore η → 1. Increasing D_eff requires changing the support; decreasing k defeats the purpose.
5. **Concentration profiles** show that in the diffusion-limited regime, reactant barely penetrates to the pellet center — the catalyst interior is wasted.

---
> **Session Break.** Good time to pause. Part 2 covers diagnostics (Weisz–Prater), reactor integration, external mass transfer, and microporous/nanotube materials.

---

---
## Part 6: Weisz--Prater Criterion

The Weisz--Prater criterion allows diagnosis of diffusion limitations using **observable quantities only**---you do not need to know the intrinsic rate constant $k$.

$$N_{WP} = \frac{r_{\text{obs}} \cdot R_p^2}{C_{A,s} \cdot D_{\text{eff}}}$$

Since $r_{\text{obs}} = \eta \cdot k \cdot C_{A,s}$ for first-order kinetics:

$$N_{WP} = \eta \cdot \phi^2$$

**Interpretation:**
- $N_{WP} < 0.3$: No significant internal diffusion limitation
- $0.3 < N_{WP} < 3$: Moderate limitation (transition regime)
- $N_{WP} > 3$: Severe diffusion limitation

In [ ]:
def weisz_prater_from_obs(r_obs, R_p, C_As, D_eff):
    """Calculate Weisz-Prater criterion from observable quantities.

    Parameters
    ----------
    r_obs : float
        Observed reaction rate (mol/(m^3 s))
    R_p : float
        Pellet radius (m)
    C_As : float
        Surface concentration of reactant (mol/m^3)
    D_eff : float
        Effective diffusivity (m^2/s)

    Returns
    -------
    float
        Weisz-Prater number (dimensionless)

    Notes
    -----
    N_WP = r_obs * R_p^2 / (C_As * D_eff)
    N_WP < 0.3  -> negligible limitation
    N_WP > 3    -> severe limitation
    """
    return r_obs * R_p**2 / (C_As * D_eff)


def weisz_prater_from_phi(phi, eta):
    """Calculate Weisz-Prater number from Thiele modulus and effectiveness factor.

    Parameters
    ----------
    phi : float or array-like
        Thiele modulus
    eta : float or array-like
        Effectiveness factor

    Returns
    -------
    float or ndarray
        Weisz-Prater number = phi^2 * eta
    """
    return np.asarray(phi)**2 * np.asarray(eta)


# --- Reproduce Example 4.3 from lecture notes ---
print("Weisz-Prater Diagnostic (Example 4.3)")
print("=" * 50)
r_obs_ex = 50.0       # mol/(m^3 s), converted from 0.05 mol/(L s)
R_p_ex = 2e-3         # m (2 mm)
C_As_ex = 500.0       # mol/m^3 (0.5 mol/L)
D_eff_ex = 5e-7       # m^2/s

N_WP = weisz_prater_from_obs(r_obs_ex, R_p_ex, C_As_ex, D_eff_ex)
print(f"r_obs   = {r_obs_ex} mol/(m^3 s)")
print(f"R_p     = {R_p_ex*1000:.0f} mm")
print(f"C_A,s   = {C_As_ex} mol/m^3")
print(f"D_eff   = {D_eff_ex:.0e} m^2/s")
print(f"\nN_WP    = {N_WP:.2f}")
if N_WP < 0.3:
    print("Diagnosis: Negligible internal diffusion limitation")
elif N_WP < 3:
    print("Diagnosis: Moderate internal diffusion limitation")
else:
    print("Diagnosis: Severe internal diffusion limitation")

In [ ]:
# Plot N_WP vs phi to show the diagnostic regions
phi_range = np.logspace(-2, 2, 500)
eta_sphere_vals = effectiveness_sphere(phi_range)
eta_slab_vals = effectiveness_slab(phi_range)

WP_sphere = weisz_prater_from_phi(phi_range, eta_sphere_vals)
WP_slab = weisz_prater_from_phi(phi_range, eta_slab_vals)

fig, ax = plt.subplots(figsize=(10, 7))

ax.loglog(phi_range, WP_sphere, color=COLORS['orange'],
          linewidth=2.5, linestyle='--', label='Sphere')
ax.loglog(phi_range, WP_slab, color=COLORS['blue'],
          linewidth=2.5, linestyle='-', label='Slab')

# Diagnostic thresholds
ax.axhline(y=0.3, color=COLORS['green'], linestyle='--',
           linewidth=2, alpha=0.8, label=r'$N_{WP}$ = 0.3 (no limitation)')
ax.axhline(y=3.0, color=COLORS['vermillion'], linestyle='--',
           linewidth=2, alpha=0.8, label=r'$N_{WP}$ = 3.0 (severe)')

# Shade diagnostic regions
ax.axhspan(0.001, 0.3, alpha=0.08, color=COLORS['green'])
ax.axhspan(3.0, 200, alpha=0.08, color=COLORS['vermillion'])
ax.text(0.015, 0.05, 'No limitation', fontsize=11,
        color=COLORS['green'], fontweight='bold')
ax.text(0.015, 50, 'Severe limitation', fontsize=11,
        color=COLORS['vermillion'], fontweight='bold')
ax.text(0.015, 0.8, 'Transition', fontsize=11,
        color='gray', fontweight='bold')

# Mark the lecture note example
ax.plot([np.nan], [N_WP], 'o', color='black', markersize=10,
        markeredgewidth=2)  # Placeholder legend

ax.set_xlabel(r'Thiele Modulus, $\phi$')
ax.set_ylabel(r'Weisz--Prater Number, $N_{WP} = \eta\phi^2$')
ax.set_title('Weisz--Prater Criterion: Diagnostic Regions')
ax.set_xlim(0.01, 100)
ax.set_ylim(0.001, 200)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

**Concept Check:** You measure N_WP = 0.8 for a first-order reaction. Your lab partner says: "We're fine — N_WP < 1, so diffusion is negligible." Do you agree?

<details><summary>Answer</summary>

**No.** For first-order kinetics, the threshold for negligible diffusion limitation is N_WP < **0.3** (corresponding to η > 0.95). At N_WP = 0.8, we are in the **transition regime** where diffusion reduces the rate by roughly 15–25%. The threshold N_WP < 1 is not the correct benchmark — it depends on reaction order (N_WP < 6 for zeroth order, < 0.3 for first order, < 0.15 for second order). Always specify the reaction order when interpreting the Weisz–Prater criterion.

</details>

---
## Part 7: PFR with Diffusion Limitation

In a real packed-bed reactor, the observed rate is reduced by the effectiveness factor:

$$r_{\text{obs}} = \eta \cdot k \cdot C_A$$

For a PFR with first-order kinetics, the effective rate constant becomes $k_{\text{eff}} = \eta \cdot k$, and the analytical solution is:

$$X = 1 - \exp(-\eta \cdot k \cdot \tau)$$

However, this simple formula uses a *constant* $\eta$, which is exact only for first-order kinetics where $\eta$ depends on $\phi$ alone (not on concentration). Let us compare PFR conversion profiles for different pellet sizes.

### Lecture Note Reference

From Example 4.2: A PFR with $k = 0.5$ s$^{-1}$ and $\tau = 4$ s achieves $X = 86.5\%$ without diffusion limitation but only $X = 55.1\%$ when $\eta = 0.4$.

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# PFR conversion with first-order kinetics and
# diffusion limitation. Change pellet radii or
# kinetic/diffusion parameters to explore behavior.
k_rxn = 0.5          # s^-1 (intrinsic rate constant)
D_eff_pfr = 1e-6     # m^2/s

# Different pellet radii to compare
R_p_values_mm = [0.1, 0.5, 1.0, 2.0, 5.0]  # mm
# =============================================

R_p_values = [r * 1e-3 for r in R_p_values_mm]  # Convert to m

tau_range = np.linspace(0, 10, 300)  # Space time in seconds

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

colors_list = [COLORS['blue'], COLORS['green'], COLORS['orange'],
               COLORS['vermillion'], COLORS['purple']]
linestyles_list = ['-', '--', '-.', '-', '--']

# Left panel: Conversion vs space time
print("PFR Conversion at tau = 4 s")
print("=" * 55)
print(f"{'R_p (mm)':>10} | {'phi':>6} | {'eta':>8} | {'X':>8}")
print("-" * 55)

for R_p, R_p_mm, color, ls in zip(R_p_values, R_p_values_mm,
                                    colors_list, linestyles_list):
    phi = thiele_modulus_sphere(R_p, k_rxn, D_eff_pfr)
    eta = effectiveness_sphere(phi)
    k_eff = eta * k_rxn
    X = 1 - np.exp(-k_eff * tau_range)

    ax1.plot(tau_range, X * 100, color=color, linestyle=ls, linewidth=2.5,
             label=f'$R_p$ = {R_p_mm} mm ($\\eta$ = {eta:.3f})')

    # Report at tau = 4 s
    X_at_4 = 1 - np.exp(-k_eff * 4)
    print(f"{R_p_mm:>10.1f} | {phi:>6.2f} | {eta:>8.4f} | {X_at_4*100:>7.1f}%")

ax1.axvline(x=4, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.set_xlabel(r'Space Time, $\tau$ (s)')
ax1.set_ylabel('Conversion, $X$ (%)')
ax1.set_title('PFR Conversion: Effect of Pellet Size')
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 100)
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Right panel: Conversion at fixed tau vs pellet radius
R_p_continuous = np.linspace(0.05e-3, 10e-3, 300)
tau_fixed_values = [2, 4, 8]  # s
tau_colors = [COLORS['blue'], COLORS['orange'], COLORS['green']]
tau_ls = ['-', '--', '-.']

for tau_fixed, tc, tls in zip(tau_fixed_values, tau_colors, tau_ls):
    phi_cont = thiele_modulus_sphere(R_p_continuous, k_rxn, D_eff_pfr)
    eta_cont = effectiveness_sphere(phi_cont)
    X_cont = 1 - np.exp(-eta_cont * k_rxn * tau_fixed)
    ax2.plot(R_p_continuous * 1000, X_cont * 100, color=tc, linestyle=tls,
             linewidth=2.5, label=f'$\\tau$ = {tau_fixed} s')

ax2.set_xlabel(r'Pellet Radius, $R_p$ (mm)')
ax2.set_ylabel('Conversion, $X$ (%)')
ax2.set_title('Conversion vs Pellet Size at Fixed Space Time')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 100)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### PFR with Concentration-Dependent Effectiveness (ODE Integration)

For non-first-order kinetics, $\eta$ depends on local concentration, and we must integrate the PFR ODE numerically:

$$\frac{dX}{d\tau} = \frac{\eta(\phi) \cdot r_{\text{intrinsic}}(C_A)}{C_{A0}}$$

Below we demonstrate this for first-order kinetics (where the analytical and numerical results should agree) and then show how to handle the general case.

In [ ]:
def pfr_ode_diffusion(tau, X, k, eta):
    """Right-hand side of PFR ODE with effectiveness factor correction.

    Parameters
    ----------
    tau : float
        Space time (s) -- independent variable
    X : array-like
        Conversion (dimensionless) -- state variable
    k : float
        Intrinsic first-order rate constant (s^-1)
    eta : float
        Effectiveness factor (constant for first-order kinetics)

    Returns
    -------
    list
        dX/dtau

    Notes
    -----
    dX/dtau = eta * k * (1 - X)  for first-order kinetics
    """
    dXdtau = eta * k * (1 - X[0])
    return [dXdtau]


# Compare analytical vs numerical for two pellet sizes
R_p_test = [0.5e-3, 3.0e-3]  # mm
tau_span = [0, 10]
tau_eval = np.linspace(0, 10, 200)

fig, ax = plt.subplots(figsize=(10, 6))

for R_p, color, ls_num, ls_ana in zip(
        R_p_test,
        [COLORS['blue'], COLORS['vermillion']],
        ['-', '-'],
        ['o', 's']):

    phi = thiele_modulus_sphere(R_p, k_rxn, D_eff_pfr)
    eta = effectiveness_sphere(phi)

    # Numerical solution
    sol = solve_ivp(pfr_ode_diffusion, tau_span, [0.0],
                    args=(k_rxn, eta), t_eval=tau_eval,
                    method='RK45', rtol=1e-8)

    # Analytical solution
    X_analytical = 1 - np.exp(-eta * k_rxn * tau_eval)

    ax.plot(sol.t, sol.y[0] * 100, color=color, linewidth=2.5,
            label=f'Numerical ($R_p$ = {R_p*1000:.1f} mm, '
                  f'$\\eta$ = {eta:.3f})')
    ax.plot(tau_eval[::20], X_analytical[::20] * 100,
            ls_ana, color=color, markersize=8,
            markerfacecolor='none', markeredgewidth=1.5,
            label=f'Analytical ($R_p$ = {R_p*1000:.1f} mm)')

ax.set_xlabel(r'Space Time, $\tau$ (s)')
ax.set_ylabel('Conversion, $X$ (%)')
ax.set_title('PFR with Diffusion Limitation: Numerical vs Analytical')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Numerical and analytical solutions agree perfectly for first-order kinetics.")
print("For other reaction orders, the ODE approach is essential.")

### Apparent Activation Energy in the Diffusion-Limited Regime

When diffusion controls ($\phi \gg 1$), the observed rate scales as:

$$r_{\text{obs}} \approx \frac{3}{\phi} \cdot k \cdot C_{A,s} = \frac{3}{R_p}\sqrt{D_{\text{eff}} \cdot k} \;\; C_{A,s}$$

Since $k \propto e^{-E_a/RT}$ and $D_{\text{eff}}$ has weak $T$ dependence, $r_{\text{obs}} \propto e^{-E_a/(2RT)}$, giving:

$$\boxed{E_{a,\text{app}} \approx \frac{E_a}{2}}$$

This halving of the apparent activation energy is a classic diagnostic: if crushing the catalyst pellet restores the full $E_a$, diffusion was limiting.

In [ ]:
# Demonstrate the halving of apparent Ea
Ea_intrinsic = 80e3   # J/mol (80 kJ/mol)
A_preexp = 1e10       # s^-1
D_eff_ea = 1e-6       # m^2/s (weak T dependence, treated as constant)
R_p_ea = 3e-3         # 3 mm

T_range_ea = np.linspace(400, 600, 50)  # K

# Intrinsic rate constant
k_intrinsic = A_preexp * np.exp(-Ea_intrinsic / (R * T_range_ea))

# Small pellet (kinetic control) -- must be small enough to stay kinetic
# across the entire T range. At 600 K, k ~ 1000 s^-1, so need R_p << sqrt(D_eff/k).
R_p_small = 0.01e-3   # 10 μm (ensures phi < 0.35 even at 600 K)
phi_small = thiele_modulus_sphere(R_p_small, k_intrinsic, D_eff_ea)
eta_small = effectiveness_sphere(phi_small)
k_obs_small = eta_small * k_intrinsic

# Large pellet (diffusion control)
phi_large = thiele_modulus_sphere(R_p_ea, k_intrinsic, D_eff_ea)
eta_large = effectiveness_sphere(phi_large)
k_obs_large = eta_large * k_intrinsic

# Arrhenius plot
fig, ax = plt.subplots(figsize=(10, 7))

inv_T = 1000 / T_range_ea  # 1000/T for readability

ax.semilogy(inv_T, k_obs_small, color=COLORS['blue'], linewidth=2.5,
            label=f'Small pellet ($R_p$ = {R_p_small*1e6:.0f} μm, kinetic control)')
ax.semilogy(inv_T, k_obs_large, color=COLORS['vermillion'], linewidth=2.5,
            label=f'Large pellet ($R_p$ = {R_p_ea*1000:.0f} mm, diffusion-limited)')

# Fit slopes for apparent Ea
from scipy.stats import linregress

# Small pellet fit
slope_s, intercept_s, _, _, _ = linregress(1/T_range_ea, np.log(k_obs_small))
Ea_app_small = -slope_s * R / 1000  # kJ/mol

# Large pellet fit
slope_l, intercept_l, _, _, _ = linregress(1/T_range_ea, np.log(k_obs_large))
Ea_app_large = -slope_l * R / 1000  # kJ/mol

# Annotation
textstr = (f'Intrinsic $E_a$ = {Ea_intrinsic/1000:.0f} kJ/mol\n'
           f'Small pellet: $E_{{a,app}}$ = {Ea_app_small:.1f} kJ/mol\n'
           f'Large pellet: $E_{{a,app}}$ = {Ea_app_large:.1f} kJ/mol\n'
           f'Ratio: {Ea_app_large/Ea_app_small:.2f} '
           f'(expected {chr(8776)} 0.5)')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.55, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

ax.set_xlabel('1000/$T$ (K$^{-1}$)')
ax.set_ylabel(r'Observed Rate Constant, $k_{obs}$ (s$^{-1}$)')
ax.set_title('Apparent Activation Energy: Kinetic vs Diffusion Control')
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

**Concept Check:** A catalyst with known intrinsic E_a = 80 kJ/mol is tested in a packed bed. You measure E_a,app = 42 kJ/mol from the Arrhenius plot. What do you conclude, and what single experiment would confirm your hypothesis?

<details><summary>Answer</summary>

The ratio E_a,app/E_a ≈ 42/80 = 0.53 ≈ 1/2, which is the classic signature of **internal diffusion limitation** (E_a,app ≈ E_a/2 when D_eff has negligible temperature dependence). To confirm: **crush the pellets** to a much smaller size and repeat. If diffusion was limiting, the crushed pellets will show (a) a higher rate and (b) E_a,app closer to 80 kJ/mol. This is the "particle size test" — the most direct experimental diagnostic for diffusion limitation.

</details>

---
## Part 8: External Mass Transfer and Combined Resistances

So far we have considered only **internal** diffusion — reactant transport *inside* the porous pellet. But there is also a **boundary layer** (film) around the pellet through which reactant must diffuse from the bulk fluid to the pellet surface.

### Diagnostics

**Carberry number** — ratio of observed rate to maximum external transfer rate:

$$Ca = \frac{r_{\text{obs}}}{k_c \, a_p \, C_{A,b}}$$

If $Ca \ll 1$, external mass transfer is negligible.

**Mears criterion** — for an $n$th-order reaction:

$$C_M = \frac{r_{\text{obs}} \, \rho_b \, R_p \, n}{k_c \, C_{A,b}} < 0.15$$

ensures less than 5% concentration drop across the film.

### Combined resistance — Biot number

The **mass transfer Biot number** compares external to internal resistance:

$$Bi_m = \frac{k_c \, R_p}{D_{\text{eff}}}$$

The **overall effectiveness factor** accounting for both resistances (sphere):

$$\eta_{\text{overall}} = \frac{\eta}{1 + \eta \, \phi^2 / (3 \, Bi_m)}$$

- $Bi_m \gg 1$: internal diffusion controls ($\eta_{\text{overall}} \approx \eta$)
- $Bi_m \ll 1$: external film controls ($\eta_{\text{overall}} \approx 3 Bi_m / \phi^2$)

In [ ]:
def overall_effectiveness(eta_internal, phi, Bi_m):
    """Overall effectiveness factor for combined internal + external resistance (sphere).

    Parameters
    ----------
    eta_internal : float or array-like
        Internal effectiveness factor (from Thiele analysis)
    phi : float or array-like
        Thiele modulus
    Bi_m : float
        Mass transfer Biot number = k_c * R_p / D_eff

    Returns
    -------
    float or ndarray
        Overall effectiveness factor
    """
    eta_internal = np.asarray(eta_internal)
    phi = np.asarray(phi)
    return eta_internal / (1 + eta_internal * phi**2 / (3 * Bi_m))


# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Compare internal-only vs combined resistance
# for different Biot numbers.
k_ext = 1.0           # s^-1
D_eff_ext = 1e-6      # m^2/s
R_p_ext = 2e-3        # 2 mm
# =============================================

phi_ext = thiele_modulus_sphere(R_p_ext, k_ext, D_eff_ext)
eta_int = effectiveness_sphere(phi_ext)

print(f"Baseline: phi = {phi_ext:.2f}, eta_internal = {eta_int:.4f}")
print()
print(f"{'Bi_m':>8} | {'eta_overall':>12} | {'eta_int':>10} | {'Dominant resistance'}")
print("-" * 60)
for Bi_m in [0.1, 1.0, 5.0, 10.0, 50.0, 1000.0]:
    eta_ov = overall_effectiveness(eta_int, phi_ext, Bi_m)
    if Bi_m < 1:
        regime = "External film"
    elif Bi_m > 10:
        regime = "Internal pore"
    else:
        regime = "Both comparable"
    print(f"{Bi_m:>8.1f} | {eta_ov:>12.4f} | {eta_int:>10.4f} | {regime}")

In [ ]:
# Plot overall eta vs phi for several Biot numbers
phi_range_ext = np.logspace(-1, 2, 400)
Bi_values = [0.5, 2, 10, 100, 1e6]
Bi_labels = ['0.5 (film controls)', '2', '10', '100', r'$\infty$ (no film)']

fig, ax = plt.subplots(figsize=(10, 7))

colors_ext = [COLORS['vermillion'], COLORS['orange'], COLORS['green'],
              COLORS['skyblue'], COLORS['blue']]
ls_ext = ['--', '-.', '-', '--', '-']

for Bi_m, label, color, ls in zip(Bi_values, Bi_labels, colors_ext, ls_ext):
    eta_int_arr = effectiveness_sphere(phi_range_ext)
    eta_ov_arr = overall_effectiveness(eta_int_arr, phi_range_ext, Bi_m)
    ax.loglog(phi_range_ext, eta_ov_arr, color=color, linestyle=ls,
              linewidth=2.5, label=f'$Bi_m$ = {label}')

ax.set_xlabel(r'Thiele Modulus, $\phi$')
ax.set_ylabel(r'Overall Effectiveness Factor, $\eta_{\mathrm{overall}}$')
ax.set_title('Combined Internal + External Resistance (Sphere)')
ax.set_xlim(0.1, 100)
ax.set_ylim(0.001, 1.5)
ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Key insight: When Bi_m is small (e.g., liquid-phase reactions),")
print("external film resistance can dominate even if internal η is high.")
print("Gas-phase reactions (high D_AB) rarely have external limitation.")

**Concept Check:** Liquid-phase Pd/C-catalyzed hydrogenation often shows external mass transfer limitation, while gas-phase CO oxidation on Pt/Al₂O₃ rarely does. Why?

<details><summary>Answer</summary>

The key is the molecular diffusivity D_AB. In **gas phase**, D_AB ~ 10⁻⁵ m²/s, so the Sherwood number gives a large k_c and thus high Bi_m (internal control). In **liquid phase**, D_AB ~ 10⁻⁹ m²/s (10,000× smaller), giving much smaller k_c and lower Bi_m. This makes external film resistance significant for liquid-phase reactions. Vigorous stirring or smaller catalyst particles help alleviate external limitation in liquid-phase systems.

</details>

---
## Part 9: Diffusion in Microporous and Nanostructured Materials

The diffusion mechanism depends on **pore size** relative to the mean free path (λ ≈ 100 nm for gases at 1 atm):

| Regime | Pore size | D_eff range | Mechanism |
|--------|-----------|-------------|-----------|
| **Molecular** | d_pore > 50 nm | 10⁻⁵–10⁻⁴ m²/s | Molecule–molecule collisions |
| **Knudsen** | 2–50 nm | 10⁻⁷–10⁻⁵ m²/s | Molecule–wall collisions |
| **Configurational** | < 2 nm | 10⁻¹⁴–10⁻⁸ m²/s | Activated "squeezing" through constrictions |

### Knudsen diffusivity

$$D_K = \frac{d_{\text{pore}}}{3} \sqrt{\frac{8RT}{\pi M}}$$

### Configurational (activated) diffusion in zeolites

$$D_{\text{conf}} = D_0 \exp\!\left(-\frac{E_D}{RT}\right)$$

where E_D = 5–100 kJ/mol depending on the molecule-to-pore size ratio.

**Important consequence for apparent E_a:** When diffusion itself is activated:

$$E_{a,\text{app}} \approx \frac{E_a + E_D}{2} \quad \text{(not just } E_a/2 \text{)}$$

This can **mask** diffusion limitation in zeolites because E_a,app stays close to E_a when E_D is large.

In [ ]:
def knudsen_diffusivity(T, M_gmol, d_pore_nm):
    """Calculate Knudsen diffusivity in a cylindrical pore.

    Parameters
    ----------
    T : float or array-like
        Temperature (K)
    M_gmol : float
        Molar mass (g/mol)
    d_pore_nm : float
        Pore diameter (nm)

    Returns
    -------
    float or ndarray
        Knudsen diffusivity (m^2/s)
    """
    d_pore_m = d_pore_nm * 1e-9
    M_kgmol = M_gmol * 1e-3
    return (d_pore_m / 3) * np.sqrt(8 * R * np.asarray(T, dtype=float) / (np.pi * M_kgmol))


def configurational_diffusivity(D0, E_diff, T):
    """Calculate configurational diffusivity in zeolite micropores.

    Parameters
    ----------
    D0 : float
        Pre-exponential factor (m^2/s)
    E_diff : float
        Diffusion activation energy (J/mol)
    T : float or array-like
        Temperature (K)

    Returns
    -------
    float or ndarray
        Configurational diffusivity (m^2/s)
    """
    return D0 * np.exp(-E_diff / (R * np.asarray(T, dtype=float)))


# Compare diffusion regimes: Knudsen vs configurational
T_compare = 500  # K

print("Diffusion Coefficient Comparison at T = 500 K")
print("=" * 65)

# Knudsen: N2 in 10 nm mesopore
D_K_N2 = knudsen_diffusivity(T_compare, 28.0, 10.0)
print(f"Knudsen (N2, 10 nm pore):        D_K = {D_K_N2:.2e} m^2/s")

# Knudsen: N2 in 1 nm micropore (overestimates -- Knudsen breaks down)
D_K_micro = knudsen_diffusivity(T_compare, 28.0, 1.0)
print(f"Knudsen (N2, 1 nm pore):         D_K = {D_K_micro:.2e} m^2/s  [upper bound]")

# Configurational: methane in FAU (E_D ~ 10 kJ/mol)
D_conf_FAU = configurational_diffusivity(1e-7, 10e3, T_compare)
print(f"Configurational (CH4/FAU):       D   = {D_conf_FAU:.2e} m^2/s  [E_D = 10 kJ/mol]")

# Configurational: isobutane in MFI (E_D ~ 45 kJ/mol)
D_conf_MFI = configurational_diffusivity(1e-7, 45e3, T_compare)
print(f"Configurational (iC4/MFI):       D   = {D_conf_MFI:.2e} m^2/s  [E_D = 45 kJ/mol]")

# Configurational: xylene in MFI (E_D ~ 80 kJ/mol)
D_conf_xyl = configurational_diffusivity(1e-7, 80e3, T_compare)
print(f"Configurational (xylene/MFI):    D   = {D_conf_xyl:.2e} m^2/s  [E_D = 80 kJ/mol]")

print(f"\nSpan: {D_K_N2/D_conf_xyl:.0e}x difference between mesopore and tight micropore!")

In [ ]:
# Apparent Ea with activated diffusion (zeolites)
# Compare: conventional support (E_D ≈ 0) vs zeolite (E_D = 50 kJ/mol)

Ea_intr = 80e3       # J/mol intrinsic activation energy
A_pre = 1e10         # s^-1
R_p_zeo = 1e-3       # 1 mm zeolite crystal
T_arr = np.linspace(400, 700, 60)

# Case 1: Mesoporous support (D_eff constant, no T dependence)
D_meso = 1e-6  # m^2/s (constant)
k_arr = A_pre * np.exp(-Ea_intr / (R * T_arr))
phi_meso = thiele_modulus_sphere(R_p_zeo, k_arr, D_meso)
eta_meso = effectiveness_sphere(phi_meso)
k_obs_meso = eta_meso * k_arr

# Case 2: Zeolite (activated diffusion, E_D = 50 kJ/mol)
E_D = 50e3  # J/mol
D0_zeo = 1e-7
D_zeo = configurational_diffusivity(D0_zeo, E_D, T_arr)
phi_zeo = thiele_modulus_sphere(R_p_zeo, k_arr, D_zeo)
eta_zeo = effectiveness_sphere(phi_zeo)
k_obs_zeo = eta_zeo * k_arr

# Case 3: Intrinsic (no diffusion)
k_obs_intr = k_arr

fig, ax = plt.subplots(figsize=(10, 7))
inv_T_arr = 1000 / T_arr

ax.semilogy(inv_T_arr, k_obs_intr, color='gray', linewidth=2, linestyle=':',
            label=f'Intrinsic ($E_a$ = {Ea_intr/1000:.0f} kJ/mol)')
ax.semilogy(inv_T_arr, k_obs_meso, color=COLORS['blue'], linewidth=2.5,
            label=f'Mesoporous ($E_D \\approx 0$)')
ax.semilogy(inv_T_arr, k_obs_zeo, color=COLORS['vermillion'], linewidth=2.5,
            linestyle='--', label=f'Zeolite ($E_D$ = {E_D/1000:.0f} kJ/mol)')

# Fit apparent Ea for each case (use high-phi region: high T)
from scipy.stats import linregress
mask_highT = T_arr > 550  # diffusion-limited region

slope_meso, _, _, _, _ = linregress(1/T_arr[mask_highT], np.log(k_obs_meso[mask_highT]))
Ea_app_meso = -slope_meso * R / 1000

slope_zeo, _, _, _, _ = linregress(1/T_arr[mask_highT], np.log(k_obs_zeo[mask_highT]))
Ea_app_zeo = -slope_zeo * R / 1000

textstr = (f'In diffusion-limited regime (T > 550 K):\n'
           f'  Mesoporous: $E_{{a,app}}$ = {Ea_app_meso:.0f} kJ/mol '
           f'(≈ $E_a$/2 = {Ea_intr/2000:.0f})\n'
           f'  Zeolite:    $E_{{a,app}}$ = {Ea_app_zeo:.0f} kJ/mol '
           f'(≈ ($E_a$+$E_D$)/2 = {(Ea_intr+E_D)/2000:.0f})')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.03, 0.05, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', bbox=props)

ax.set_xlabel('1000/$T$ (K$^{-1}$)')
ax.set_ylabel(r'Observed Rate Constant, $k_{\mathrm{obs}}$ (s$^{-1}$)')
ax.set_title('Apparent $E_a$: Conventional vs Activated Diffusion')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Key insight: For the zeolite, E_a,app = {Ea_app_zeo:.0f} kJ/mol is much closer")
print(f"to E_a = {Ea_intr/1000:.0f} kJ/mol than to E_a/2 = {Ea_intr/2000:.0f} kJ/mol.")
print(f"This can MASK diffusion limitation! Use the Weisz-Prater criterion instead.")

In [ ]:
# CNT Enhanced Transport: How enhancement factor F affects η
# D_CNT = F × D_Knudsen, where F = 10 to 10,000 for CNTs

# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
k_cnt = 10.0             # s^-1 (fast reaction)
L_cnt = 5e-6             # 5 μm membrane thickness
D_K_classical = 1e-6     # m^2/s (classical Knudsen)
# =============================================

# Classical Thiele modulus (no enhancement)
phi_classical = L_cnt * np.sqrt(k_cnt / D_K_classical)

F_values = [1, 10, 100, 1000, 10000]

print("CNT Enhancement Factor Effect on Effectiveness")
print("=" * 60)
print(f"Baseline: phi_classical = {phi_classical:.3f}")
print(f"{'F':>8} | {'phi_eff':>8} | {'eta':>8} | {'Regime'}")
print("-" * 60)
for F in F_values:
    phi_eff = phi_classical / np.sqrt(F)
    eta_F = effectiveness_slab(phi_eff)
    regime = "kinetic" if phi_eff < 0.3 else ("diffusion" if phi_eff > 3 else "transition")
    print(f"{F:>8,d} | {phi_eff:>8.4f} | {eta_F:>8.4f} | {regime}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: η vs F
F_range = np.logspace(0, 4, 200)
phi_eff_range = phi_classical / np.sqrt(F_range)
eta_F_range = effectiveness_slab(phi_eff_range)

ax1.semilogx(F_range, eta_F_range, color=COLORS['blue'], linewidth=2.5)
ax1.axhline(y=0.95, color='gray', linestyle='--', linewidth=1)
ax1.set_xlabel('Enhancement Factor, $F$')
ax1.set_ylabel(r'Effectiveness Factor, $\eta$')
ax1.set_title('CNT Enhancement Restores Effectiveness')
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

# Find F needed for η > 0.95
idx_95_F = np.argmin(np.abs(eta_F_range - 0.95))
F_needed = F_range[idx_95_F]
ax1.annotate(f'$F$ ≈ {F_needed:.0f} for $\\eta$ > 0.95',
             xy=(F_needed, 0.95), xytext=(F_needed * 5, 0.75),
             fontsize=11, arrowprops=dict(arrowstyle='->', lw=1.2))

# Right: Knudsen D_K vs CNT diameter
d_cnt_range = np.linspace(1, 20, 100)  # nm
D_K_range = knudsen_diffusivity(500, 28.0, d_cnt_range)

ax2.plot(d_cnt_range, D_K_range * 1e6, color=COLORS['green'], linewidth=2.5,
         label='Classical Knudsen')
ax2.plot(d_cnt_range, D_K_range * 1e6 * 100, color=COLORS['orange'],
         linewidth=2.5, linestyle='--', label='CNT ($F$ = 100)')
ax2.plot(d_cnt_range, D_K_range * 1e6 * 1000, color=COLORS['vermillion'],
         linewidth=2.5, linestyle='-.', label='CNT ($F$ = 1000)')
ax2.set_xlabel('Pore/Tube Diameter (nm)')
ax2.set_ylabel(r'Diffusivity ($\times 10^{-6}$ m$^2$/s)')
ax2.set_title('Knudsen vs CNT-Enhanced Diffusivity (N$_2$, 500 K)')
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nKey insight: CNTs can enhance transport by F = 10-10,000 over Knudsen,")
print(f"dramatically reducing the effective Thiele modulus and restoring η → 1.")

**Concept Check:** A zeolite catalyst with E_a = 100 kJ/mol and E_D = 60 kJ/mol shows E_a,app = 78 kJ/mol. A student concludes "no diffusion limitation since E_a,app is close to E_a." Is this correct?

<details><summary>Answer</summary>

**No!** With activated diffusion, E_a,app ≈ (E_a + E_D)/2 = (100 + 60)/2 = 80 kJ/mol, which is close to the measured 78 kJ/mol. The system **is** diffusion-limited, but the activated nature of micropore diffusion keeps E_a,app close to E_a, masking the limitation. The E_a/2 diagnostic only works for conventional (non-activated) diffusion. For zeolites, **always use the Weisz–Prater criterion** as the primary diagnostic.

</details>

### Key Takeaways — Part 2 (Diagnostics, External Transfer, and Special Materials)

1. **Weisz–Prater criterion** (N_WP = r_obs·R_p²/(C_As·D_eff)) diagnoses diffusion limitation from observable quantities. Thresholds depend on reaction order: < 0.3 (1st), < 6 (0th), < 0.15 (2nd).
2. **Apparent Ea halving** (E_a,app ≈ E_a/2) is a classic diagnostic — but only for non-activated diffusion. Always confirm with the particle size test.
3. **External film resistance** matters for liquid-phase reactions (low D_AB → low Bi_m). Gas-phase reactions rarely have external limitation.
4. **Combined resistance**: η_overall = η/(1 + η·φ²/(3·Bi_m)). When Bi_m ≫ 1, internal controls; when Bi_m ≪ 1, external controls.
5. **Zeolites**: Configurational diffusion is activated (E_D = 5–100 kJ/mol), giving E_a,app ≈ (E_a + E_D)/2, which can mask diffusion limitation.
6. **CNTs**: Enhancement factors F = 10–10,000 reduce the effective Thiele modulus, restoring η → 1 even for very fast reactions.

---
## Exercises

### Exercise 1: Finding the Critical Thiele Modulus

At what value of $\phi$ does the effectiveness factor drop below 0.5 for (a) a slab and (b) a sphere?

**Task:** Write code to find $\phi_{0.5}$ numerically for each geometry.

In [ ]:
# YOUR CODE HERE
# Hint: Create a fine array of phi values and find where eta crosses 0.5.
# You can use np.argmin(np.abs(eta - 0.5)) to find the index.
#
# (a) Slab: phi_05_slab = ?
# (b) Sphere: phi_05_sphere = ?
#
# Expected: slab phi ~ 1.9, sphere phi ~ 4.7

pass  # Replace with your implementation

### Exercise 2: Pellet Design for Target Effectiveness

A first-order catalytic reaction has $k = 5.0$ s$^{-1}$ and $D_{\text{eff}} = 2 \times 10^{-6}$ m$^2$/s on a spherical pellet. Design the maximum pellet radius $R_p$ that achieves $\eta > 0.95$.

**Task:** Calculate the required maximum $R_p$ and verify by plotting $\eta$ vs $R_p$.

In [ ]:
# YOUR CODE HERE
# Given: k = 5.0 s^-1, D_eff = 2e-6 m^2/s
# Target: eta > 0.95 (sphere geometry)
#
# Step 1: Find phi that gives eta_sphere = 0.95
# Step 2: Calculate R_p = phi * sqrt(D_eff / k)
# Step 3: Plot eta vs R_p with the target marked

pass  # Replace with your implementation

### Exercise 3: Diffusion Limitation at High vs Low Conversion

Does diffusion limitation reduce conversion more at high $\tau$ (targeting high conversion) or at low $\tau$ (targeting low conversion)?

**Task:** Compare the *absolute reduction in conversion* ($\Delta X = X_{\eta=1} - X_{\eta<1}$) at $\tau = 1$ s and $\tau = 10$ s for a PFR with $k = 0.5$ s$^{-1}$ and $\eta = 0.4$. Explain the result physically.

**Hint:** Use the first-order PFR formula $X = 1 - \exp(-\eta k \tau)$.

In [ ]:
# YOUR CODE HERE
# Calculate X_ideal (eta=1) and X_limited (eta=0.4) at tau = 1 s and 10 s
# Compare the absolute difference Delta_X = X_ideal - X_limited
#
# Also create a plot of Delta_X vs tau for the full range

pass  # Replace with your implementation

### Exercise 4: External vs Internal Limitation

A liquid-phase hydrogenation over Pd/C pellets (R_p = 1 mm) has:
- r_obs = 20 mol/(m³·s), C_A,s = 100 mol/m³
- D_eff = 5 × 10⁻⁷ m²/s (internal), k_c = 2 × 10⁻⁴ m/s (external)

**Tasks:**
1. Calculate N_WP. Is internal diffusion significant?
2. Calculate the Biot number Bi_m. Is external film resistance significant?
3. Calculate η_overall and compare it to the internal η alone.
4. Suggest two practical changes to improve the overall effectiveness.

In [ ]:
# YOUR CODE HERE
# Given: r_obs = 20 mol/(m^3 s), C_As = 100 mol/m^3
#        D_eff = 5e-7 m^2/s, k_c = 2e-4 m/s, R_p = 1e-3 m
#
# 1. N_WP = r_obs * R_p^2 / (C_As * D_eff)
# 2. Bi_m = k_c * R_p / D_eff
# 3. Use overall_effectiveness() with appropriate phi and eta
# 4. Consider: smaller pellets, faster stirring, different solvent

pass  # Replace with your implementation

### Exercise 5: Zeolite vs Mesoporous Catalyst — Thiele Modulus Comparison

Compare the effectiveness factor for n-hexane cracking (k = 50 s⁻¹ at 773 K) on:
- **H-ZSM-5 zeolite crystal** (R_p = 0.5 μm, D_eff = 5 × 10⁻¹² m²/s via configurational diffusion)
- **Pt/γ-Al₂O₃ pellet** (R_p = 1 mm, D_eff = 5 × 10⁻⁷ m²/s via Knudsen diffusion)

**Tasks:**
1. Calculate φ and η for each catalyst.
2. Which catalyst has a higher effectiveness factor despite having far lower D_eff? Explain why.
3. For the zeolite, what maximum crystal radius would maintain η > 0.9?

**Hint:** The zeolite crystal size is 10⁶× smaller than the pellet, compensating for the 10⁵× smaller diffusivity.

In [ ]:
# YOUR CODE HERE
# Zeolite: R_p = 0.5e-6 m, D_eff = 5e-12 m^2/s, k = 50 s^-1
# Pellet:  R_p = 1e-3 m,   D_eff = 5e-7 m^2/s,  k = 50 s^-1
#
# Calculate phi and eta for each using thiele_modulus_sphere and effectiveness_sphere.
# For part 3: find R_p where eta = 0.9 (invert the eta formula or use numerical search).

pass  # Replace with your implementation

---
## Reflection Questions

1. The Thiele modulus includes both $k$ and $D_{\text{eff}}$. Physically, why does *increasing* the intrinsic rate constant $k$ make the catalyst *less* effective (i.e., lower $\eta$)?

2. Industrial catalysts for automotive exhaust treatment use thin washcoats on monolith supports rather than packed beds of large pellets. Based on the analysis in this notebook, explain the engineering rationale.

3. The apparent activation energy drops from $E_a$ to approximately $E_a/2$ in the diffusion-limited regime. How could you use this observation as a diagnostic test in the laboratory? What experiment would you perform?

4. For the Weisz--Prater criterion, all quantities ($r_{\text{obs}}$, $R_p$, $C_{A,s}$, $D_{\text{eff}}$) are in principle measurable. Which of these is typically the most difficult to determine accurately, and why?

5. Zeolite catalysts can have $E_{a,\text{app}} \approx (E_a + E_D)/2$ instead of $E_a/2$. Explain why the standard $E_a$-halving diagnostic fails for microporous materials, and name a more reliable diagnostic.

6. Liquid-phase reactions are far more susceptible to *external* mass transfer limitation than gas-phase reactions. Explain why, relating your answer to the Biot number.

**Next notebook:** NB5 covers selectivity engineering (Chapter 5).